# 03. Lists, dicts, and tuples

Welcome to the third exercise of the course. The first two notebooks worked with a single list and a single number; this one steps up to the three workhorse Python data structures used together. The goal is to feel at home with lists, dictionaries, and tuples: how to build them, append to them, merge them, convert between them, and recognize which one is the right tool for a given moment.

This exercise is denser than the previous two. Plan on closer to forty minutes than thirty if everything is new, or stay closer to thirty if these data structures are already familiar from another language.

Readers who completed exercises 01 and 02 will recognize the data and the framing. Readers who skipped them can pick up here, but the `for` loop, `if`/`elif`/`else`, `def`, and `import` are taken for granted and not explained again.

The task is a Sales Plan restatement. Last year's monthly sales sit in a list, and the month names are held as a tuple. Late in the year, accounting submitted corrections for two months. The job is to:

1. Build a dict that maps each month name to last year's sales figure for that month, by pairing the tuple of month names with the list of values.
2. Apply the corrections by merging a small dict of restatements into the base dict, with the corrections overriding the originals where the keys match.
3. Convert the merged dict into a list of `(month, value)` tuples in calendar order, and append a final `("Year", total)` record so the list reads as a small report.
4. Print the report.

Five operations show up across the steps, one per pillar of the topic:

- **Building dicts from sequences.** `dict(zip(keys, values))` pairs two sequences position by position and turns the result into a dict.
- **Merging dicts.** The `|` operator combines two dicts into a new one; on key collisions, the right side wins. `d1 | d2` was added to dict in Python 3.9 and is the modern equivalent of `{**d1, **d2}`.
- **Converting between types.** Lists, tuples, and dicts interoperate. `list(some_tuple)` and `tuple(some_list)` produce the corresponding sequence type; `dict.items()` gives an iterable of `(key, value)` tuples; `dict(some_iterable_of_pairs)` goes back the other way.
- **Appending to a list.** `records.append(item)` adds an element to the end of a list, in place. Lists are mutable; the operation modifies the list that already exists rather than producing a new one.
- **Tuple immutability.** A tuple cannot be modified after it is created. `MONTHS.append(...)` raises `AttributeError`; `MONTHS[0] = ...` raises `TypeError`. The reason for using a tuple here, rather than a list, is precisely that the calendar months are not meant to change during the run of the program.

The notebook is arranged in four tiers, ordered from least to most support:

- **Pro.** The brief above is the whole instruction. A blank code cell follows the data, and the rest is yours.
- **Advanced.** A short description of the approach and the tools used, plus a code cell with a comment outline for the functions.
- **Beginner.** Five collapsible hints, then a code cell with the function shapes already in place. Slots marked `<...>` are the parts to fill in.
- **Solution.** The full working code with a brief explanation of the choices it makes.

Read the notebook from top to bottom. Start at whichever tier feels right today, drop down a tier if a higher one stalls, and treat the Solution as a reference rather than a finish line.

Before retrying the exercise at a different tier in the same session, restart the Jupyter kernel and run the Data cell again. Function names defined at one tier carry into the next and would otherwise mask the slots in the next tier.

Documentation pointers:

- Lists: https://docs.python.org/3/tutorial/introduction.html#lists
- Dictionaries: https://docs.python.org/3/tutorial/datastructures.html#dictionaries
- Tuples and sequences: https://docs.python.org/3/tutorial/datastructures.html#tuples-and-sequences
- The `zip` built-in: https://docs.python.org/3/library/functions.html#zip
- Dict union with `|` (PEP 584): https://docs.python.org/3/whatsnew/3.9.html#pep-584-add-union-operators-to-dict

The exercise is pure Python. No tm1py call is made. The framing is the same Sales Plan model used in the previous exercises.

## Data

Three values drive the exercise: a tuple of month names, a list of last year's monthly sales figures, and a small dict of restatements.

`MONTHS` is a tuple of twelve strings, ordered from January to December. The names are short ("Jan", "Feb", and so on). It is written as a tuple rather than a list because the names of the months do not change during the run of the program, and tuples are the natural Python type for fixed sequences. Anyone who tried to call `MONTHS.append("XIII")` would get an `AttributeError`, which is exactly the safety the type provides.

`sales` is the same list of twelve floats as in the previous exercises: monthly sales for last year, ordered to match `MONTHS`. The pairing of `MONTHS` and `sales` is what every later step builds on.

`restated` is a small dict carrying two corrections from accounting. Each key is a month name; each value is the corrected figure for that month. The original `sales` list is left untouched; the corrections will be folded into a separate dict at task time, by merging.

If the kernel was restarted between tiers, run the cell below again before continuing. The names `MONTHS`, `sales`, and `restated` are referenced by the Pro, Advanced, Beginner, and Solution code cells.

In [ ]:
MONTHS: tuple[str, ...] = ("Jan", "Feb", "Mar", "Apr", "May", "Jun",
                           "Jul", "Aug", "Sep", "Oct", "Nov", "Dec")
sales: list[float] = [120.0, 95.0, 110.0, 130.0, 80.0, 105.0,
                      140.0, 100.0, 115.0, 125.0, 90.0, 135.0]
restated: dict[str, float] = {"Jul": 145.0, "Aug": 88.0}

## Pro

This tier is for readers who want to take the brief cold and write the program from a blank cell.

The data is already defined just above; `MONTHS`, `sales`, and `restated` are ready to use. The brief is at the top of the notebook. Read it, decide on the function shapes, and write the program in the cell below. There is no scaffolding here, and that is the point of the tier.

If the cell stops being productive to stare at, scroll on to Advanced or Beginner. The lower tiers do not spoil the work that has already gone into this one.

## Advanced

This tier is for readers who can see the rough shape of the solution and want a small amount of structure to write into.

The approach is built from three small functions and a `main` that ties them together.

`to_monthly` takes a tuple of month names and a list of values and returns a dict mapping each month name to the value at the same position. The natural way to write it is `dict(zip(months, values))`. The `zip` built-in pairs two sequences index by index, and `dict(...)` consumes the iterable of pairs and produces a dictionary.

`apply_restatements` takes a base dict and a corrections dict and returns a new dict with the corrections layered on top. The natural way to write it is `base | corrections`. The `|` operator returns a new dict that contains all keys from both sides; where a key appears in both, the right side wins. Neither input dict is modified.

`to_records` takes the merged dict and the tuple of month names and returns a list of `(month, value)` tuples in calendar order. Iterate over `MONTHS` rather than over the dict directly. Iterating over a dict yields keys in insertion order, which would happen to match in this case, but iterating over `MONTHS` makes the intention explicit and survives changes to the data later. After the per month tuples are built, the function appends one more entry: `("Year", total)`, where `total` is the sum of the merged values.

`main` calls the three in order and then prints each record on its own line. Each record is a `(month, value)` tuple, and tuple unpacking in the `for` header (`for month, value in records:`) makes the print line read naturally.

A few remarks on conversion and immutability that are worth absorbing while writing the code:

- A list and a tuple of the same elements are interchangeable in most read contexts. `list(some_tuple)` returns a list with the same elements; `tuple(some_list)` returns a tuple. Both conversions are shallow; they do not deeply copy the elements inside.
- `dict.items()` returns a view of `(key, value)` pairs. The view is iterable but is not a list. Wrapping it in `list(...)` materializes a list of tuples, which is sometimes useful for sorting or slicing.
- Tuples are immutable. A tuple cannot be modified in place; `MONTHS[0] = "January"` raises `TypeError`, and `MONTHS.append(...)` raises `AttributeError` because the method does not exist. The route to a "modified" tuple is to convert to a list, modify the list, and convert back, which in effect produces a new tuple.

The cell below has a comment outline for the three helper functions and `main`. Fill in the bodies and call `main()` on the last line.

In [ ]:
# Function: to_monthly. Given a tuple of month names and a list of values,
# return a dict mapping each month name to the value at the same position.

# Function: apply_restatements. Given a base dict and a corrections dict,
# return a new dict where the corrections override the base on shared keys.

# Function: to_records. Given a merged dict and the tuple of month names,
# return a list of (month, value) tuples in calendar order, with a final
# ("Year", total) tuple appended.

# Function: main. Build base, apply restatements, build records, print each record.

# Last line: call main().


## Beginner

This tier is for readers who want guided support: hints to read one at a time, and a code skeleton with the function shapes already in place.

Open the hints below in order, only as far as needed. Each hint addresses one operation. Below the hints, a code cell holds the function signatures and the body structure already written; the slots marked `<...>` are the parts to fill in. The slot names are descriptive and suggest what belongs in each one.

<details><summary>1. How is a dict built from two sequences?</summary>

`zip(months, values)` pairs the two sequences position by position and yields an iterable of two element tuples: `("Jan", 120.0)`, `("Feb", 95.0)`, and so on. Wrapping it in `dict(...)` consumes that iterable and produces a dictionary: `dict(zip(months, values))`. The same pattern works whenever there are two sequences of the same length, one carrying keys and one carrying values.

</details>

<details><summary>2. How are two dicts merged?</summary>

`base | corrections` returns a new dict that contains all keys from both sides. Where a key appears in both, the value from the right side wins. Neither `base` nor `corrections` is modified. The `|` operator on dict was added in Python 3.9; before that, the equivalent expression was `{**base, **corrections}`.

</details>

<details><summary>3. How are records iterated in a guaranteed order?</summary>

Iterate over the tuple `MONTHS` rather than over the dict. `for month in MONTHS:` walks the tuple in calendar order, and `merged[month]` looks up the value for that month. This gives calendar order regardless of how the dict was built. For each iteration, build a `(month, merged[month])` tuple and add it to a list.

</details>

<details><summary>4. How is a list of tuples built, and how is a final entry appended?</summary>

A list comprehension is the most natural way to build the per month list: `[(month, monthly[month]) for month in months]`. A list created this way is mutable, so `records.append(("Year", total))` adds the final entry at the end. The total is `sum(monthly.values())`, or equivalently `sum(monthly[month] for month in months)`.

</details>

<details><summary>5. Why is `MONTHS` a tuple, and not a list?</summary>

The tuple `MONTHS` is meant to be set once and never modified; using a tuple makes that intention explicit and prevents accidental modification. Calling `MONTHS.append(...)` raises `AttributeError`, because tuples have no `append` method. Assigning to an index, `MONTHS[0] = "Jan."`, raises `TypeError`, because tuples do not support item assignment. The route to a "modified" tuple is `tuple(list(MONTHS) + [extra])` or similar; in practice, the more honest move is to keep month names in a tuple and never modify them.

</details>

In [ ]:
def to_monthly(months: tuple[str, ...], values: list[float]) -> dict[str, float]:
    return <CONSTRUCTOR_FOR_DICT>(<PAIRING_OF_MONTHS_AND_VALUES>)

def apply_restatements(base: dict[str, float], corrections: dict[str, float]) -> dict[str, float]:
    return base <OPERATOR_FOR_MERGE> corrections

def to_records(monthly: dict[str, float], months: tuple[str, ...]) -> list[tuple[str, float]]:
    records: list[tuple[str, float]] = [(month, monthly[month]) for month in months]
    total = <BUILTIN_FOR_SUM>(monthly.values())
    records.<METHOD_TO_ADD_AT_END>((<LABEL_FOR_TOTAL>, total))
    return records

def main() -> None:
    base = to_monthly(MONTHS, sales)
    merged = apply_restatements(base, restated)
    records = to_records(merged, MONTHS)
    for month, value in records:
        print(month, value)

main()

## Solution

The cell below holds a complete, working version of the exercise. As before, this is offered as a reference rather than as the only correct form; alternative approaches that arrive at the same output are equally good.

The structure is three small helper functions plus `main`. `to_monthly` pairs the tuple of month names with the list of values via `zip`, then turns the resulting iterable into a dict in a single expression. `apply_restatements` returns the union of the two dicts using the `|` operator; the original `base` and `corrections` dicts are not modified. `to_records` walks `MONTHS` in calendar order to build a list of `(month, value)` tuples, then computes the year total and appends a final `("Year", total)` record. `main` calls the three in order and prints each record on its own line.

A few notes on the choices made:

- `MONTHS` is iterated explicitly to guarantee calendar order in the report. Iterating over the merged dict directly would happen to give the same order in this case, because the base dict was built from `MONTHS` and the corrections dict only updated existing keys. The explicit iteration is more readable and survives changes to the data later.
- The total is computed with `sum(monthly.values())`, which sums the values of the dict regardless of key order. The sum is over twelve numbers, so the cost is negligible; on larger data, a single sum call is also more efficient than building the records first and summing them again.
- The `for month, value in records:` line uses tuple unpacking. Each record is a two element tuple, and the `for` header pulls the two pieces apart by position. The same idea works wherever a sequence yields fixed length tuples; it is one of the small ergonomics that make tuples comfortable to work with.

For a feel of how the data flows through these operations, copy this code into a debugger (the VS Code Python debugger, or `pdb` from the terminal) and step through it line by line. Watch the shape of `base` after `to_monthly` returns, the shape of `merged` after the `|` merge, and the growing `records` list as the comprehension and the final `append` execute. Pandas is not yet on stage, but the same habit pays off even more once it is: seeing the intermediate values is what turns operations into intuition.

In [ ]:
def to_monthly(months: tuple[str, ...], values: list[float]) -> dict[str, float]:
    return dict(zip(months, values))

def apply_restatements(base: dict[str, float], corrections: dict[str, float]) -> dict[str, float]:
    return base | corrections

def to_records(monthly: dict[str, float], months: tuple[str, ...]) -> list[tuple[str, float]]:
    records: list[tuple[str, float]] = [(month, monthly[month]) for month in months]
    total = sum(monthly.values())
    records.append(("Year", total))
    return records

def main() -> None:
    base = to_monthly(MONTHS, sales)
    merged = apply_restatements(base, restated)
    records = to_records(merged, MONTHS)
    for month, value in records:
        print(month, value)

main()